# MaritimeBench: End-to-End Benchmark Orchestration & Artifact Inventory Pipeline
**Canonical Interactive Benchmark Execution Workflow**
*Authoritative Corpus: Corpus A (MARSIS / TSB Canadian Maritime Corpus)*

---

### Pipeline Architecture & Execution Protocol
This notebook serves as the **orchestration and artifact-inventory layer** for the complete MaritimeBench pipeline (Stages 01 through 18).
Production scripts remain responsible for their own data and scientific schema logic. Notebook validation is intentionally limited to process success, artifact existence, artifact non-zero size, and runtime metadata:

1. **Modular Script Execution**: Invokes canonical, environment-agnostic production scripts in scripts/ sequentially via subprocess.
2. **Sequential Stage Gates**: Each stage strictly asserts that the preceding stage returned exit code 0 before proceeding.
3. **Basic Artifact Inventory**: Validates expected output file/directory existence, non-zero size, and non-empty directory item counts.
4. **Safe Resumption**: Detects pre-existing, valid stage outputs to avoid redundant recomputations while allowing explicit reruns (RESUME_IF_VALID policy).
5. **Hardware & Preflight Checks**: Verifies required uploaded directories (scripts/, data/, config/, 	emplates/) and enforces GPU/CUDA gating prior to Stage 14.
6. **Remediated Benchmark Capabilities**:
   - **Dynamic Candidate Cohort**: Retains all non-redundant candidates (including microsoft/deberta-v3-base) dynamically derived from Stage 13.
   - **A01 MLM Masking Robustness**: Evaluates baseline Subword Masking (Mode 1, default) with optional Whole-Word Masking (Modes 2 & 3).
   - **Final Artifact Inventory**: Generates comprehensive machine-readable (maritimebench_output_inventory.json) and human-readable (maritimebench_output_inventory.md) reports.

In [ ]:
# Cell 1: Environment Detection & Reproducibility Metadata
import sys
import os
import platform
import subprocess
import importlib.util
from pathlib import Path
from datetime import datetime, timezone

print("=" * 70)
print("MARITIMEBENCH BENCHMARK ORCHESTRATOR - PREFLIGHT & RUNTIME INFO")
print("=" * 70)

# Project root detection (robust for local repo and Google Colab)
def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, current.parent, current.parent.parent]:
        if (candidate / "scripts").exists() and (candidate / "data").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
print(f"Detected Project Root: {PROJECT_ROOT}")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

# Dependency Preflight Verification
REQUIRED_PACKAGES = [
    ("torch", "torch"),
    ("transformers", "transformers"),
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("scipy", "scipy"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("sklearn", "scikit-learn"),
    ("tqdm", "tqdm")
]

# Explicit dependency installation policy:
# False: Report missing packages and exact installation command; do NOT modify environment silently.
# True: Automatically install missing declared requirements.
AUTO_INSTALL_DEPENDENCIES = False

missing_packages = []
for mod_name, pkg_name in REQUIRED_PACKAGES:
    if importlib.util.find_spec(mod_name) is None:
        missing_packages.append((mod_name, pkg_name))

if missing_packages:
    install_cmd = f"pip install {' '.join(p[1] for p in missing_packages)}"
    print("\n" + "!" * 70)
    print(f"[PREFLIGHT WARNING] Missing {len(missing_packages)} required package(s):")
    for m, p in missing_packages:
        print(f"  * {m} (install via: {p})")
    print(f"Recommended installation command:\n  {install_cmd}")
    print("!" * 70)
    if AUTO_INSTALL_DEPENDENCIES:
        print("[INFO] AUTO_INSTALL_DEPENDENCIES is True. Installing declared requirements...")
        subprocess.run([sys.executable, "-m", "pip", "install", *[p[1] for p in missing_packages]], check=True)
        print("[OK] Missing packages installed.")
    else:
        raise ImportError(
            f"Missing required packages: {[p[0] for p in missing_packages]}. "
            f"Run '{install_cmd}' or set AUTO_INSTALL_DEPENDENCIES = True."
        )
else:
    print("[OK] Required packages preflight check passed.")

# Record Runtime Metadata
py_ver = sys.version.split()[0]
os_info = f"{platform.system()} {platform.release()} ({platform.machine()})"
timestamp = datetime.now(timezone.utc).isoformat()

# Git Commit Detection
git_commit = "NOT_A_GIT_REPO"
try:
    git_proc = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True)
    git_commit = git_proc.stdout.strip()
except Exception:
    pass

# Deep Learning Frameworks
torch_ver, cuda_avail, gpu_name, gpu_mem = "NOT_INSTALLED", False, "NONE", "NONE"
try:
    import torch
    torch_ver = torch.__version__
    cuda_avail = torch.cuda.is_available()
    if cuda_avail:
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB"
except ImportError:
    pass

transformers_ver = "NOT_INSTALLED"
try:
    import transformers
    transformers_ver = transformers.__version__
except ImportError:
    pass

print(f"Timestamp (UTC)      : {timestamp}")
print(f"Git Commit SHA       : {git_commit}")
print(f"Python Version       : {py_ver}")
print(f"Operating System     : {os_info}")
print(f"PyTorch Version      : {torch_ver}")
print(f"CUDA Available       : {cuda_avail}")
print(f"GPU Hardware         : {gpu_name} (Memory: {gpu_mem})")
print(f"HuggingFace Transformers: {transformers_ver}")
print("=" * 70)

In [ ]:
# Cell 2: Preflight Verification for Uploaded Dependencies & Authoritative Corpus
import hashlib

def calculate_sha256(filepath: Path) -> str:
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

print("Verifying uploaded directory structure and Corpus A files...")

REQUIRED_DIRECTORIES = ["scripts", "data", "config", "templates"]
REQUIRED_DATA_FILES = [
    "data/MARSISdb_MDOTW_VW_INJURIES_PUBLIC.csv",
    "data/MARSISdb_MDOTW_VW_OCCURRENCE_PUBLIC.csv",
    "data/MARSISdb_MDOTW_VW_OCCURRENCE_VESSEL_LSA_EQUIPMENT_PUBLIC.csv",
    "data/MARSISdb_MDOTW_VW_OCCURRENCE_VESSEL_NAV_EQUIPMENT_PUBLIC.csv",
    "data/MARSISdb_MDOTW_VW_OCCURRENCE_VESSEL_PUBLIC.csv",
    "data/MARSISdb_MDOTW_VW_OCCURRENCE_VESSEL_REC_EQUIPMENT_PUBLIC.csv",
    "data/MDOTW-MARSIS-Master-dataset-inventory-and-dictionary-English.csv"
]
REQUIRED_CONFIG_TEMPLATES = [
    ("config/config.json", "Defines pipeline output directory and base parameters."),
    ("templates/equipment_templates.json", "Required by Stage 06 to synthesize equipment narrative sentences."),
    ("templates/injury_templates.json", "Required by Stage 06 to synthesize injury narrative sentences."),
    ("templates/vessel_templates.json", "Required by Stage 06 to synthesize vessel narrative sentences.")
]

missing_items = []
for d in REQUIRED_DIRECTORIES:
    dir_path = PROJECT_ROOT / d
    if not dir_path.is_dir():
        missing_items.append((d, "Required top-level pipeline directory."))

for rel_f in REQUIRED_DATA_FILES:
    f_path = PROJECT_ROOT / rel_f
    if not f_path.is_file():
        missing_items.append((rel_f, "Authoritative Corpus A relational table."))

for rel_f, reason in REQUIRED_CONFIG_TEMPLATES:
    f_path = PROJECT_ROOT / rel_f
    if not f_path.is_file():
        missing_items.append((rel_f, reason))

if missing_items:
    print("\n[ERROR] PREFLIGHT VERIFICATION FAILED! Missing required dependencies:")
    for path_str, reason in missing_items:
        print(f"  * MISSING REQUIRED FILE/DIR: {path_str}")
        print(f"    WHY REQUIRED: {reason}")
    raise FileNotFoundError("Preflight verification failed. Please upload all missing files before proceeding.")
else:
    print("[SUCCESS] Minimal upload verified: scripts/ + data/ + config/ + templates/")
    print(f"Found all {len(REQUIRED_DATA_FILES)} Corpus A data tables.")
    # Log occurrence file hash for auditable provenance
    occ_csv = PROJECT_ROOT / "data/MARSISdb_MDOTW_VW_OCCURRENCE_PUBLIC.csv"
    occ_sha = calculate_sha256(occ_csv)
    print(f"Corpus A Primary Occurrence Table SHA256: {occ_sha[:16]}...{occ_sha[-16:]}")

# Ensure standard project outputs directory exists
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Standard outputs directory established: {OUTPUTS_DIR}")


In [ ]:
# Cell 3: Orchestration Engine, Output Inventory Helpers & Execution Logic
import sys
import os
import time
import subprocess
from pathlib import Path
from typing import Optional, List, Dict, Any

# Execution Policy Configuration:
# 'RESUME_IF_VALID': Reuses existing stage outputs if all expected artifacts exist and are non-empty
# 'ALWAYS_RERUN': Forces re-execution of all production scripts
EXECUTION_POLICY = "RESUME_IF_VALID"

# MLM Evaluation Mode Configuration (Stage 14):
# MODE 1 (Baseline Benchmark):
#   MLM_MASKING_MODE = "subword"
#   MLM_EVALUATION_UNIT = "subword"
# MODE 2 (Whole-Word Masking Subword Diagnostic):
#   MLM_MASKING_MODE = "whole_word"
#   MLM_EVALUATION_UNIT = "subword"
# MODE 3 (Whole-Word Masking Strict Word Reconstruction):
#   MLM_MASKING_MODE = "whole_word"
#   MLM_EVALUATION_UNIT = "word"
MLM_MASKING_MODE = "subword"
MLM_EVALUATION_UNIT = "subword"

# Stage 14 Compute Policy:
# The full Stage 14 benchmark requires a CUDA-enabled GPU.
# CPU execution is allowed ONLY for lightweight smoke testing / compatibility verification.
ALLOW_CPU_SMOKE_TEST = False

# Global Pipeline Results Tracker
STAGE_RESULTS = []

class BenchmarkStageException(Exception):
    """Raised when a stage fails execution or artifact inventory check."""
    pass

def format_size(size_bytes: int) -> str:
    """Formats byte count into human-readable string."""
    if size_bytes < 1024:
        return f"{size_bytes} B"
    elif size_bytes < 1024 ** 2:
        return f"{size_bytes / 1024:.1f} KB"
    elif size_bytes < 1024 ** 3:
        return f"{size_bytes / (1024 ** 2):.1f} MB"
    else:
        return f"{size_bytes / (1024 ** 3):.2f} GB"

def inventory_artifact(target_path: Path) -> Dict[str, Any]:
    """Checks a single expected output path for existence, type, and size."""
    try:
        rel_path = str(target_path.relative_to(PROJECT_ROOT))
    except Exception:
        rel_path = str(target_path)

    if not target_path.exists():
        return {
            "path": rel_path,
            "exists": False,
            "type": "MISSING",
            "size_bytes": 0,
            "size_display": "0 B",
            "items_count": 0
        }

    if target_path.is_file():
        sz = target_path.stat().st_size
        return {
            "path": rel_path,
            "exists": True,
            "type": "FILE",
            "size_bytes": sz,
            "size_display": format_size(sz),
            "items_count": 1
        }
    elif target_path.is_dir():
        items = list(target_path.glob("*"))
        total_sz = sum(f.stat().st_size for f in items if f.is_file())
        return {
            "path": rel_path,
            "exists": True,
            "type": "DIRECTORY",
            "size_bytes": total_sz,
            "size_display": format_size(total_sz),
            "items_count": len(items)
        }
    return {
        "path": rel_path,
        "exists": False,
        "type": "UNKNOWN",
        "size_bytes": 0,
        "size_display": "0 B",
        "items_count": 0
    }

def inventory_outputs(expected_paths: List[Path]) -> List[Dict[str, Any]]:
    """Inventories a list of expected artifact paths."""
    return [inventory_artifact(p) for p in expected_paths]

def run_stage(
    stage_id: str,
    script_rel_path: str,
    expected_outputs: Optional[List[Path]] = None,
    extra_args: Optional[List[str]] = None
):
    print(f"\n{'=' * 70}")
    print(f"ORCHESTRATING {stage_id.upper()}: {script_rel_path}")
    print(f"{'=' * 70}")

    script_path = PROJECT_ROOT / script_rel_path
    if not script_path.exists():
        raise BenchmarkStageException(f"Production script not found: {script_rel_path}")

    # Check safe resumption: if all expected outputs already exist and are non-empty
    if EXECUTION_POLICY == "RESUME_IF_VALID" and expected_outputs:
        inv = inventory_outputs(expected_outputs)
        all_present = all(
            item["exists"] and (
                (item["type"] == "FILE" and item["size_bytes"] > 0) or
                (item["type"] == "DIRECTORY" and item["items_count"] > 0)
            )
            for item in inv
        )
        if all_present:
            print(f"[RESUME] Existing valid output artifacts detected for {stage_id}.")
            for item in inv:
                print(f"  * {item['path']} ({item['type']}, {item['size_display']}, {item['items_count']} items)")
            resumed_entry = {
                "stage": stage_id,
                "script": script_rel_path,
                "return_code": 0,
                "status": "RESUMED (OUTPUTS_EXIST)",
                "runtime_seconds": 0.0,
                "artifacts": inv
            }
            STAGE_RESULTS.append(resumed_entry)
            return resumed_entry

    cmd = [sys.executable, str(script_path)]
    if extra_args:
        cmd.extend(extra_args)

    print(f"Executing: {' '.join(cmd)}")
    t0 = time.time()
    proc = subprocess.run(cmd, capture_output=True, text=True, cwd=PROJECT_ROOT)
    elapsed = time.time() - t0

    if proc.returncode != 0:
        print("\n" + "=" * 70)
        print(f"FAILED STAGE: {stage_id}")
        print(f"ERROR: Subprocess exited with non-zero exit code: {proc.returncode}")
        print(f"COMMAND: {' '.join(cmd)}")
        print(f"RUNTIME: {elapsed:.2f}s")
        print("=" * 70)
        print("STDERR TRACEBACK:")
        print(proc.stderr if proc.stderr else "(No stderr output)")
        print("=" * 70)
        print("STDOUT TAIL:")
        stdout_lines = proc.stdout.splitlines()
        print("\n".join(stdout_lines[-25:]) if stdout_lines else "(No stdout output)")
        print("=" * 70)
        STAGE_RESULTS.append({
            "stage": stage_id,
            "script": script_rel_path,
            "return_code": proc.returncode,
            "status": f"FAILED (Exit Code {proc.returncode})",
            "runtime_seconds": round(elapsed, 2),
            "artifacts": []
        })
        raise BenchmarkStageException(f"Stage {stage_id} execution failed with exit code {proc.returncode}.")

    # Script exited with code 0: Verify output artifact existence and non-zero size
    inv = inventory_outputs(expected_outputs) if expected_outputs else []
    missing_or_empty = []
    for item in inv:
        if not item["exists"]:
            missing_or_empty.append(f"{item['path']} (MISSING)")
        elif item["type"] == "FILE" and item["size_bytes"] == 0:
            missing_or_empty.append(f"{item['path']} (ZERO BYTES)")
        elif item["type"] == "DIRECTORY" and item["items_count"] == 0:
            missing_or_empty.append(f"{item['path']} (EMPTY DIRECTORY)")

    if missing_or_empty:
        print("\n" + "=" * 70)
        print(f"ARTIFACT INVENTORY FAILURE: {stage_id}")
        print("Process exited with code 0 but expected output artifacts were missing or zero-sized:")
        for m in missing_or_empty:
            print(f"  * {m}")
        print("=" * 70)
        STAGE_RESULTS.append({
            "stage": stage_id,
            "script": script_rel_path,
            "return_code": proc.returncode,
            "status": "FAILED (MISSING_ARTIFACTS)",
            "runtime_seconds": round(elapsed, 2),
            "artifacts": inv
        })
        raise BenchmarkStageException(f"Stage {stage_id} missing expected outputs: {missing_or_empty}")

    print(f"[PASS] {stage_id} completed successfully in {elapsed:.2f}s.")
    print("Generated Artifacts:")
    for item in inv:
        print(f"  * {item['path']} ({item['type']}, {item['size_display']})")

    result_entry = {
        "stage": stage_id,
        "script": script_rel_path,
        "return_code": 0,
        "status": "PASS",
        "runtime_seconds": round(elapsed, 2),
        "artifacts": inv
    }
    STAGE_RESULTS.append(result_entry)
    return result_entry

print("=" * 70)
print("[OK] ORCHESTRATION ENGINE & ARTIFACT INVENTORY INITIALIZED")
print(f"  * Execution Policy    : {EXECUTION_POLICY}")
print(f"  * MLM Masking Mode    : {MLM_MASKING_MODE}")
print(f"  * Evaluation Unit     : {MLM_EVALUATION_UNIT}")
print(f"  * Allow CPU Smoke Test: {ALLOW_CPU_SMOKE_TEST}")
print("  * Orchestrator Mode   : Pure Sequential Execution + Artifact Inventory")
print("=" * 70)

## Stage 01: Data Dictionary Parsing & Schema Extraction
Parses data inventory dictionary to establish authoritative column metadata and table schemas.


In [ ]:
# Stage 01: Data Dictionary Parsing & Schema Extraction
run_stage(
    stage_id="Stage 01",
    script_rel_path="scripts/01_parse_dictionary.py",
    expected_outputs=[
        OUTPUTS_DIR / "stage-01/dictionary_metadata.json"
    ]
)

## Stage 02: Dataset Profiling & Distribution Analysis
Profiles data types, missingness rates, and cardinality across all raw relational tables.


In [ ]:
# Stage 02: Dataset Profiling & Distribution Analysis
run_stage(
    stage_id="Stage 02",
    script_rel_path="scripts/02_profile_dataset.py",
    expected_outputs=[
        OUTPUTS_DIR / "stage-02/profiling_report.json"
    ]
)

## Stage 03: Relational Key Discovery
Discovers foreign keys, occurrence joins, and cardinality graphs across tables.


In [ ]:
# Stage 03: Relational Key Discovery
run_stage(
    stage_id="Stage 03",
    script_rel_path="scripts/03_discover_relationships.py",
    expected_outputs=[
        OUTPUTS_DIR / "stage-03/relationships.json"
    ]
)

## Stage 04: Semantic Column Selection
Selects narrative, categorical, and technical attributes for downstream document synthesis.


In [ ]:
# Stage 04: Semantic Column Selection
run_stage(
    stage_id="Stage 04",
    script_rel_path="scripts/04_select_semantic_columns.py",
    expected_outputs=[
        OUTPUTS_DIR / "stage-04/selected_semantic_columns.json"
    ]
)

## Stage 05 & Stage 05a: Table Joining & Record Validation
Joins secondary relational tables to primary occurrences and validates relational integrity.


In [ ]:
# Stage 05 & Stage 05a: Table Joining & Record Validation
run_stage(
    stage_id="Stage 05",
    script_rel_path="scripts/05_merge_tables.py",
    expected_outputs=[
        OUTPUTS_DIR / "stage-05/merged_records.jsonl",
        OUTPUTS_DIR / "stage-05/merge_reconciliation_report.json"
    ]
)

run_stage(
    stage_id="Stage 05a",
    script_rel_path="scripts/05a_validate_records.py",
    expected_outputs=[
        OUTPUTS_DIR / "stage-05a/validation_report.json"
    ]
)

## Stage 06: Document Generation
Synthesizes raw maritime technical documents across multiple structural representations.


In [ ]:
# Stage 06: Document Generation
run_stage(
    stage_id="Stage 06",
    script_rel_path="scripts/06_generate_documents.py",
    expected_outputs=[
        OUTPUTS_DIR / "stage-06/raw_documents.jsonl"
    ]
)

## Stage 07: Document Cleaning & Normalization
Applies domain-aware sanitization, Unicode normalization, and whitespace cleanup.


In [ ]:
# Stage 07: Document Cleaning & Normalization
run_stage(
    stage_id="Stage 07",
    script_rel_path="scripts/07_clean_documents.py",
    expected_outputs=[
        OUTPUTS_DIR / "stage-07/clean_documents.jsonl"
    ]
)

## Stage 08: Corpus Export & Canonical Manifest
Exports canonical Corpus A (`maritime_corpus.jsonl` and `.txt`) with cryptographic manifest.


In [ ]:
# Stage 08: Corpus Export & Canonical Manifest
run_stage(
    stage_id="Stage 08",
    script_rel_path="scripts/08_export_corpus.py",
    expected_outputs=[
        OUTPUTS_DIR / "stage-08/maritime_corpus.jsonl",
        OUTPUTS_DIR / "stage-08/maritime_corpus.txt",
        OUTPUTS_DIR / "stage-08/manifest.json"
    ]
)

## Stage 09: Statistical Profiling of Canonical Corpus
Computes word length distributions, vocabulary richness, and lexical density.


In [ ]:
# Stage 09: Statistical Profiling of Canonical Corpus
run_stage(
    stage_id="Stage 09",
    script_rel_path="scripts/09_statistics.py",
    expected_outputs=[
        OUTPUTS_DIR / "stage-09/statistics.json",
        OUTPUTS_DIR / "stage-09/corpus_quality_report.md"
    ]
)

## Stage 10: Maritime Vocabulary Extraction
Extracts authoritative maritime domain terminology (335 terms) and rare maritime terms.


In [ ]:
# Stage 10: Maritime Vocabulary Extraction
run_stage(
    stage_id="Stage 10",
    script_rel_path="scripts/10_extract_vocabulary.py",
    expected_outputs=[
        OUTPUTS_DIR / "stage-10/maritime_vocabulary.txt"
    ]
)

## Stage 11: Multi-Representation Generation
Constructs the 5 canonical structural representations:
`narrative`, `key_value`, `template`, `json`, `mixed`.


In [ ]:
# Stage 11: Multi-Representation Generation
run_stage(
    stage_id="Stage 11",
    script_rel_path="scripts/11_corpus_representations.py",
    expected_outputs=[
        OUTPUTS_DIR / "stage-11/corpus_representations"
    ]
)

## Stage 12: Semantic Importance Stratification & Subset Partitioning
Stratifies corpus into 5 knowledge subsets (`high_knowledge`, `medium_knowledge`, `low_knowledge`, `balanced_knowledge`, `random_baseline`) and a General English baseline.


In [ ]:
# Stage 12: Semantic Importance Stratification & Subset Partitioning
run_stage(
    stage_id="Stage 12",
    script_rel_path="scripts/12_semantic_importance.py",
    expected_outputs=[
        OUTPUTS_DIR / "stage-12/document_importance.jsonl",
        OUTPUTS_DIR / "stage-12/subsets"
    ]
)

## Stage 13: Tokenizer Analysis & Candidate Model Selection
Evaluates candidate tokenizers across the corpus. Dynamically identifies non-redundant model architectures with fully auditable selection logging (`candidate_selection_audit.json` / `.csv`).

In [ ]:
# Stage 13: Tokenizer Analysis & Candidate Model Selection
run_stage(
    stage_id="Stage 13",
    script_rel_path="scripts/13_tokenizer_analysis.py",
    expected_outputs=[
        OUTPUTS_DIR / "stage-13/selected_models.json",
        OUTPUTS_DIR / "stage-13/candidate_selection_audit.json",
        OUTPUTS_DIR / "stage-13/candidate_selection_audit.csv",
        OUTPUTS_DIR / "stage-13/tokenizer_analysis"
    ]
)

## Stage 14: Masked Language Model (MLM) Benchmarking & Robustness (A01)
Evaluates pretrained encoders across the factorial experimental grid:
$$\text{Total Runs} = N_{\text{models}} \times R_{\text{representations}} \times S_{\text{subsets}}$$
- **Dynamic Cohort**: Dynamically calculates expected factorial cells ($N \times R \times S$) and pairwise combinations ($N(N-1)/2$).
- **A01 Robustness Support**: Supports Mode 1 (`--masking_mode subword --evaluation_unit subword`, default baseline), Mode 2 (`--masking_mode whole_word --evaluation_unit subword`), and Mode 3 (`--masking_mode whole_word --evaluation_unit word`).
- **Hardware Preflight**: Full benchmark requires CUDA GPU. CPU execution is strictly restricted to smoke-test compatibility mode.

In [ ]:
# Stage 14: Masked Language Model (MLM) Benchmarking & Robustness
import torch
import json

# 1. Dynamic Benchmark Planning / Logging (for display and planning only, not schema validation)
selected_models_path = OUTPUTS_DIR / "stage-13/selected_models.json"
if selected_models_path.exists():
    try:
        with open(selected_models_path, "r", encoding="utf-8") as f:
            raw_c = json.load(f)
        selected_cohort = raw_c.get("selected_models", raw_c) if isinstance(raw_c, dict) else raw_c
    except Exception:
        selected_cohort = []
else:
    selected_cohort = []

reps_dir = OUTPUTS_DIR / "stage-11/corpus_representations"
discovered_reps = sorted([p.stem for p in reps_dir.glob("*.jsonl") if p.is_file()]) if reps_dir.exists() else []

subsets_dir = OUTPUTS_DIR / "stage-12/subsets"
canonical_candidates = ["high_knowledge", "medium_knowledge", "low_knowledge", "balanced_knowledge", "random_baseline"]
discovered_subsets = sorted([s for s in canonical_candidates if (subsets_dir / f"{s}.jsonl").exists()]) if subsets_dir.exists() else []

N = len(selected_cohort)
R = len(discovered_reps)
S = len(discovered_subsets)
expected_cells = N * R * S
expected_pairs = N * (N - 1) // 2

print(f"Selected models (N)       : {N} -> {selected_cohort}")
print(f"Representations (R)       : {R} -> {discovered_reps}")
print(f"Canonical Subsets (S)     : {S} -> {discovered_subsets}")
print(f"Expected Factorial Cells  : {N} x {R} x {S} = {expected_cells} cells")
print(f"Expected Pairwise Comps   : {N}*({N}-1)/2 = {expected_pairs} pairs")
print(f"Configured Masking Mode   : {MLM_MASKING_MODE}")
print(f"Configured Evaluation Unit: {MLM_EVALUATION_UNIT}")

# 2. GPU Safety Gate
cuda_available = torch.cuda.is_available()
if not cuda_available:
    print("\n" + "!" * 70)
    print("[HARDWARE POLICY] CUDA is NOT available on this system.")
    print(f"Running full Stage 14 ({expected_cells} cells) on CPU is computationally infeasible.")
    print("In Google Colab, please switch to a GPU runtime: Runtime -> Change runtime type -> T4 / A100 GPU.")
    if ALLOW_CPU_SMOKE_TEST:
        print("[POLICY] ALLOW_CPU_SMOKE_TEST is enabled. Executing lightweight smoke test only.")
        stage14_args = ["--smoke-test"]
        stage14_mode = "SMOKE_TEST_ONLY"
    else:
        print("STATUS: BLOCKED_GPU_REQUIRED. Halting execution.")
        print("To test pipeline orchestration locally on CPU, set ALLOW_CPU_SMOKE_TEST = True.")
        print("!" * 70)
        raise BenchmarkStageException(f"BLOCKED_GPU_REQUIRED: Full Stage 14 benchmark requires CUDA GPU.")
else:
    gpu_name = torch.cuda.get_device_name(0)
    print(f"[OK] CUDA GPU detected: {gpu_name}. Ready for full benchmark execution.")
    stage14_args = [
        "--masking_mode", MLM_MASKING_MODE,
        "--evaluation_unit", MLM_EVALUATION_UNIT,
        "--device", "cuda"
    ]
    stage14_mode = "FULL_BENCHMARK_GPU"

# 3. Stage 14 Execution & Artifact Inventory
run_stage(
    stage_id="Stage 14",
    script_rel_path="scripts/14_mlm_evaluation.py",
    expected_outputs=[
        OUTPUTS_DIR / "stage-14/evaluations",
        OUTPUTS_DIR / "stage-14/pll_results.json",
        OUTPUTS_DIR / "stage-14/pll_selection.json"
    ],
    extra_args=stage14_args
)

## Stage 15: Cross-Model Multi-Attribute Benchmarking & MUI Evaluation
Computes Maritime Utility Index (MUI), Pareto-optimal frontiers, and multi-attribute leaderboards.
*Note: Evaluates existing production implementation as established by reconciled audit.*


In [ ]:
# Stage 15: Cross-Model Multi-Attribute Benchmarking & MUI Evaluation
run_stage(
    stage_id="Stage 15",
    script_rel_path="scripts/15_cross_model_benchmarking.py",
    expected_outputs=[
        OUTPUTS_DIR / "stage-15/leaderboard.csv",
        OUTPUTS_DIR / "stage-15/comparison.csv",
        OUTPUTS_DIR / "stage-15/stage15_model_profiles.csv",
        OUTPUTS_DIR / "stage-15/stage15_pareto.csv",
        OUTPUTS_DIR / "stage-15/stage15_mui_sensitivity.csv",
        OUTPUTS_DIR / "stage-15/stage15_report.md"
    ]
)

## Stage 16: Statistical Significance & Ablation Analysis
Performs bootstrap resampling, pairwise hypothesis testing, and weight sensitivity ablations.


In [ ]:
# Stage 16: Statistical Significance & Ablation Analysis
run_stage(
    stage_id="Stage 16",
    script_rel_path="scripts/16_statistical_analysis.py",
    expected_outputs=[
        OUTPUTS_DIR / "stage-16/stage16_pairwise_tests.csv",
        OUTPUTS_DIR / "stage-16/stage16_global_tests.csv",
        OUTPUTS_DIR / "stage-16/stage16_bootstrap.csv",
        OUTPUTS_DIR / "stage-16/ablation_study.json",
        OUTPUTS_DIR / "stage-16/stage16_final_report.md"
    ]
)

## Stage 17: Decision Engine & Leaderboard Recommendation
Applies multi-criteria decision rules, generating deployment recommendations and executive reports.


In [ ]:
# Stage 17: Decision Engine & Leaderboard Recommendation
run_stage(
    stage_id="Stage 17",
    script_rel_path="scripts/17_decision_engine.py",
    expected_outputs=[
        OUTPUTS_DIR / "stage-17/decision_summary.json",
        OUTPUTS_DIR / "stage-17/stage17_model_selection.csv",
        OUTPUTS_DIR / "stage-17/benchmark_report.md"
    ]
)

## Stage 18: Final Corpus Linting & Integrity Audit
Audits output consistency, document length constraints, and vocabulary compliance.


In [ ]:
# Stage 18: Final Corpus Linting & Integrity Audit
run_stage(
    stage_id="Stage 18",
    script_rel_path="scripts/18_lint_corpus.py",
    expected_outputs=[
        OUTPUTS_DIR / "stage-18/corpus_lint_report.json"
    ]
)

## Final Output Inventory & Benchmark Summary Report
Aggregates execution logs, runtimes, artifact counts, and validation statuses across all stages.


In [ ]:
# Final Output Inventory & Run Report
import json
from datetime import datetime, timezone
import pandas as pd

print("\n" + "=" * 90)
print("MARITIMEBENCH BENCHMARK - FINAL EXECUTION INVENTORY REPORT")
print("=" * 90)

inventory_summary = []
for entry in STAGE_RESULTS:
    stage_name = entry["stage"]
    status = entry["status"]
    runtime = f"{entry['runtime_seconds']:.1f}s"
    artifacts = entry.get("artifacts", [])
    art_count = len(artifacts)
    art_summary = ", ".join([f"{a['path'].split('/')[-1]} ({a['size_display']})" for a in artifacts[:2]])
    if art_count > 2:
        art_summary += f", ... (+{art_count - 2} more)"
    inventory_summary.append({
        "Stage": stage_name,
        "Script": entry["script"].split("/")[-1],
        "Status": status,
        "Runtime": runtime,
        "Artifacts": art_summary
    })

df_inventory = pd.DataFrame(inventory_summary)
print(df_inventory.to_string(index=False))
print("=" * 90)

# Build Detailed Output Inventory Report
all_passed = all("PASS" in e["status"] or "RESUMED" in e["status"] for e in STAGE_RESULTS)
final_pipeline_status = "SUCCESS" if all_passed else "INCOMPLETE_OR_FAILED"

inventory_report_data = {
    "report_type": "MaritimeBench Execution & Artifact Inventory Report",
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "git_commit": git_commit if "git_commit" in globals() else "UNKNOWN",
    "environment": os_info if "os_info" in globals() else "UNKNOWN",
    "python_version": py_ver if "py_ver" in globals() else "UNKNOWN",
    "torch_version": torch_ver if "torch_ver" in globals() else "UNKNOWN",
    "cuda_available": cuda_avail if "cuda_avail" in globals() else False,
    "gpu_name": gpu_name if "gpu_name" in globals() else "NONE",
    "masking_mode": MLM_MASKING_MODE if "MLM_MASKING_MODE" in globals() else "subword",
    "evaluation_unit": MLM_EVALUATION_UNIT if "MLM_EVALUATION_UNIT" in globals() else "subword",
    "pipeline_status": final_pipeline_status,
    "stages": STAGE_RESULTS
}

# Export Machine-Readable Inventory JSON
inv_json_path = OUTPUTS_DIR / "maritimebench_output_inventory.json"
with open(inv_json_path, "w", encoding="utf-8") as f_json:
    json.dump(inventory_report_data, f_json, indent=2)
print(f"[OK] Machine-readable inventory report exported to: {inv_json_path.relative_to(PROJECT_ROOT)}")

# Export Human-Readable Inventory Markdown
md_lines = [
    "# MaritimeBench Execution & Artifact Inventory Report",
    "",
    f"- **Timestamp (UTC)**: `{inventory_report_data['timestamp']}`",
    f"- **Git Commit SHA**: `{inventory_report_data['git_commit']}`",
    f"- **Environment**: `{inventory_report_data['environment']}`",
    f"- **Python / PyTorch**: `{inventory_report_data['python_version']}` / `{inventory_report_data['torch_version']}`",
    f"- **CUDA Available**: `{inventory_report_data['cuda_available']}` ({inventory_report_data['gpu_name']})",
    f"- **MLM Masking Mode / Evaluation Unit**: `{inventory_report_data['masking_mode']}` / `{inventory_report_data['evaluation_unit']}`",
    f"- **Overall Pipeline Status**: **{final_pipeline_status}**",
    "",
    "## Stage-by-Stage Artifact Inventory",
    ""
]

for entry in STAGE_RESULTS:
    md_lines.append(f"### {entry['stage']}: {entry['script']} ({entry['status']}, {entry['runtime_seconds']:.1f}s)")
    if entry.get("artifacts"):
        for a in entry["artifacts"]:
            md_lines.append(f"- `{a['path']}`: **{a['type']}**, {a['size_display']} ({a['items_count']} items)")
    else:
        md_lines.append("- *(No artifacts recorded)*")
    md_lines.append("")

inv_md_path = OUTPUTS_DIR / "maritimebench_output_inventory.md"
with open(inv_md_path, "w", encoding="utf-8") as f_md:
    f_md.write("\n".join(md_lines))
print(f"[OK] Human-readable inventory report exported to: {inv_md_path.relative_to(PROJECT_ROOT)}")

print("\n" + "=" * 60)
print("MANUAL ARTIFACT INSPECTION INVENTORY")
print("=" * 60)
for entry in STAGE_RESULTS:
    print(f"{entry['stage']}: {entry['status']}")
    for a in entry.get("artifacts", []):
        print(f"  - {a['path'].split('/')[-1]:<35} ({a['size_display']})")
print("=" * 60)

if all_passed:
    print("[ALL STAGES VERIFIED] All pipeline stages executed and output artifacts inventoried successfully!")
else:
    print("[WARNING] One or more stages encountered failure or missing outputs.")